In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_PATH = "../data/raw/ethiopia_fi_unified_data.csv"
REF_PATH = "../data/raw/reference_codes.csv"

df = pd.read_csv(DATA_PATH)
ref = pd.read_csv(REF_PATH)

print(f"Unified dataset: {df.shape[0]} records, {df.shape[1]} columns")
print(f"Reference codes: {ref.shape[0]} valid (field, code) pairs")

Unified dataset: 62 records, 35 columns
Reference codes: 71 valid (field, code) pairs


In [2]:
df["record_type"].value_counts()

record_type
observation    32
impact_link    16
event          11
target          3
Name: count, dtype: int64

In [3]:
df.head(3)

,record_id,parent_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,NaN,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaN,NaN,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Baseline year,NaN
1,REC_0002,NaN,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaN,NaN,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN
2,REC_0003,NaN,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN


In [4]:
for col in ["record_type", "pillar", "source_type", "confidence"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- record_type ---
record_type
observation    32
impact_link    16
event          11
target          3
Name: count, dtype: int64

--- pillar ---
pillar
ACCESS           23
USAGE            18
NaN              11
GENDER            6
AFFORDABILITY     4
Name: count, dtype: int64

--- source_type ---
source_type
NaN           16
operator      15
survey        10
regulator      7
research       6
policy         4
calculated     2
news           2
Name: count, dtype: int64

--- confidence ---
confidence
high      47
medium    14
low        1
Name: count, dtype: int64


In [5]:
events = df[df["record_type"] == "event"]
events[["record_id", "category", "indicator", "observation_date"]].sort_values("observation_date")

,record_id,category,indicator,observation_date
33,EVT_0001,product_launch,Telebirr Launch,2021-05-17
41,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01
34,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01
35,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01
36,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01
37,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29
38,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01
59,EVT_0011,regulation,Banking Business Proclamation No. 1360/2024 (F...,2024-12-17
39,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27
42,EVT_0010,pricing,Safaricom Ethiopia Price Increase,2025-12-15


In [6]:
categorical_fields = ["record_type", "category", "pillar", "indicator_direction",
                       "value_type", "source_type", "confidence", "gender", "location",
                       "relationship_type", "impact_direction", "impact_magnitude",
                       "evidence_basis"]

violations = []
for field in categorical_fields:
    valid_codes = set(ref.loc[ref["field"] == field, "code"])
    actual = df[field].dropna()
    actual = actual[actual != ""]
    bad = sorted(set(actual) - valid_codes)
    if bad:
        violations.append((field, bad))

if violations:
    for field, bad in violations:
        print(f"INVALID values in '{field}': {bad}")
else:
    print("All categorical values match reference_codes.csv. No schema violations found.")

All categorical values match reference_codes.csv. No schema violations found.


In [7]:
obs = df[df["record_type"] == "observation"].copy()
obs["observation_date"] = pd.to_datetime(obs["observation_date"], errors="coerce")
print("Observation date range:", obs["observation_date"].min(), "to", obs["observation_date"].max())

Observation date range: 2014-12-31 00:00:00 to 2025-12-31 00:00:00


In [8]:
coverage = (
    obs.groupby("indicator_code")
       .agg(indicator=("indicator", "first"),
            pillar=("pillar", "first"),
            n_obs=("record_id", "count"),
            first_year=("fiscal_year", "min"),
            last_year=("fiscal_year", "max"))
       .sort_values("pillar")
)
coverage

,indicator,pillar,n_obs,first_year,last_year
indicator_code,,,,,
ACC_4G_COV,4G Population Coverage,ACCESS,2,FY2022/23,FY2024/25
ACC_ELECTRICITY,Electricity Access Rate,ACCESS,1,2023,2023
ACC_FAYDA,Fayda Digital ID Enrollment,ACCESS,3,2024,2025
ACC_MM_ACCOUNT,Mobile Money Account Rate,ACCESS,2,2021,2024
ACC_MOBILE_PEN,Mobile Subscription Penetration,ACCESS,1,2025,2025
ACC_OWNERSHIP,Account Ownership Rate,ACCESS,6,2014,2024
ACC_SMARTPHONE,Smartphone Adoption Rate,ACCESS,1,2023,2023
AFF_DATA_INCOME,Data Affordability Index,AFFORDABILITY,1,2024,2024
GEN_GAP_MOBILE,Mobile Phone Gender Gap,GENDER,1,2024,2024


In [9]:
events_view = events[["record_id", "category", "indicator", "observation_date", "source_name", "confidence"]]
events_view

,record_id,category,indicator,observation_date,source_name,confidence
33,EVT_0001,product_launch,Telebirr Launch,2021-05-17,Ethio Telecom,high
34,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01,News,high
35,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01,Safaricom,high
36,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01,NIDP,high
37,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29,NBE,high
38,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01,EthSwitch,high
39,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27,EthSwitch,high
40,EVT_0008,infrastructure,EthioPay Instant Payment System Launch,2025-12-18,NBE/EthSwitch,high
41,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01,NBE,high
42,EVT_0010,pricing,Safaricom Ethiopia Price Increase,2025-12-15,News,high


In [10]:
links = df[df["record_type"] == "impact_link"].copy()
links_joined = links.merge(
    events[["record_id", "indicator"]].rename(columns={"record_id": "parent_id", "indicator": "event_name"}),
    on="parent_id", how="left"
)
links_joined[["record_id", "parent_id", "event_name", "related_indicator",
              "relationship_type", "impact_direction", "impact_magnitude",
              "impact_estimate", "lag_months", "evidence_basis", "comparable_country"]]

,record_id,parent_id,event_name,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country
0,IMP_0001,EVT_0001,Telebirr Launch,ACC_OWNERSHIP,direct,increase,high,15.0,12.0,literature,Kenya
1,IMP_0002,EVT_0001,Telebirr Launch,USG_TELEBIRR_USERS,direct,increase,high,NaN,3.0,empirical,NaN
2,IMP_0003,EVT_0001,Telebirr Launch,USG_P2P_COUNT,direct,increase,high,25.0,6.0,empirical,NaN
3,IMP_0004,EVT_0002,Safaricom Ethiopia Commercial Launch,ACC_4G_COV,direct,increase,medium,15.0,12.0,empirical,NaN
4,IMP_0005,EVT_0002,Safaricom Ethiopia Commercial Launch,AFF_DATA_INCOME,indirect,decrease,medium,-20.0,12.0,literature,Rwanda
5,IMP_0006,EVT_0003,M-Pesa Ethiopia Launch,USG_MPESA_USERS,direct,increase,high,NaN,3.0,empirical,NaN
6,IMP_0007,EVT_0003,M-Pesa Ethiopia Launch,ACC_MM_ACCOUNT,direct,increase,medium,5.0,6.0,theoretical,NaN
7,IMP_0008,EVT_0004,Fayda Digital ID Program Rollout,ACC_OWNERSHIP,enabling,increase,medium,10.0,24.0,literature,India
8,IMP_0009,EVT_0004,Fayda Digital ID Program Rollout,GEN_GAP_ACC,indirect,decrease,medium,-5.0,24.0,literature,India
9,IMP_0010,EVT_0005,Foreign Exchange Liberalization,AFF_DATA_INCOME,indirect,increase,high,30.0,3.0,empirical,NaN


In [11]:
new_ids = ["REC_0034", "REC_0035", "EVT_0011", "IMP_0015", "IMP_0016"]
df[df["record_id"].isin(new_ids)][
    ["record_id", "record_type", "pillar", "indicator", "value_numeric",
     "confidence", "source_name", "collected_by", "collection_date"]
]

,record_id,record_type,pillar,indicator,value_numeric,confidence,source_name,collected_by,collection_date
57,REC_0034,observation,ACCESS,Smartphone Adoption Rate,40.0,high,GSMA Mobile Economy Sub-Saharan Africa 2024,Ekram Kemer,2026-07-17
58,REC_0035,observation,ACCESS,Electricity Access Rate,55.4,high,World Bank SDG7 / ESMAP Tracking SDG7 Electrif...,Ekram Kemer,2026-07-17
59,EVT_0011,event,NaN,Banking Business Proclamation No. 1360/2024 (F...,NaN,high,Ethiopian Parliament / Legal500,Ekram Kemer,2026-07-17
60,IMP_0015,impact_link,ACCESS,Foreign Bank Entry effect on Account Ownership,NaN,medium,NaN,Ekram Kemer,2026-07-17
61,IMP_0016,impact_link,USAGE,Foreign Bank Entry effect on Digital Payment A...,NaN,low,NaN,Ekram Kemer,2026-07-17
